In [ ]:
from typing import Tuple

import os
import sys
import subprocess
import pickle
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

sys.path.append(os.path.abspath("../"))
sys.path.append(os.path.abspath("../symbolic_scripts"))
from symbolic_scripts.expected_number_of_subgraphs import expected_number_of_cycle_subgraphs, expected_number_of_clique_subgraphs, expected_number_of_bipartite_complete_subgraphs

In [ ]:
COLORS_MAPPING = {
    2: mcolors.TABLEAU_COLORS['tab:red'],
    3: mcolors.TABLEAU_COLORS['tab:blue'],
    4: mcolors.TABLEAU_COLORS['tab:orange'],
    5: mcolors.TABLEAU_COLORS['tab:green'],
    6: mcolors.TABLEAU_COLORS['tab:purple'],
}

#### Utilities

In [ ]:
def generate_ba_graph_nx(n0: int, n: int, m: int) -> nx.Graph | None:
    tmp_filename = "temporary-graph.txt"

    try:
        result = subprocess.run(
            ["../build/generate_graph", str(5), str(n0), str(m), str(n), tmp_filename],
            check=True,
        )
    except subprocess.CalledProcessError as e:
        print(f"Error generating graph: {e}")
        os.remove(tmp_filename) if os.path.exists(tmp_filename) else None
        return None

    if result.returncode != 0:
        print(f"Graph generation failed with return code {result.returncode}")
        os.remove(tmp_filename) if os.path.exists(tmp_filename) else None
        return None
    
    try:
        with open(tmp_filename, "r") as f:
            graph_data = f.read()
    except Exception as e:
        print(f"Error reading file {tmp_filename}: {e}")
        os.remove(tmp_filename) if os.path.exists(tmp_filename) else None
        return None

    graph = nx.Graph()
    for line in graph_data.splitlines():
        node1, node2 = [int(node) for node in line.strip().split()]
        graph.add_edge(node1, node2)

    return graph

In [ ]:
def save_graph_nx(graph: nx.Graph, filename: str) -> bool:
    try:
        with open(filename, "w") as f:
            for edge in graph.edges():
                f.write(f"{edge[0] + 1} {edge[1] + 1}\n")
        return True
    except Exception as e:
        print(f"Error saving graph to {filename}: {e}")
        return False


In [ ]:
def count_subgraphs(graph: nx.Graph, subgraph: nx.Graph, nthreads: int) -> int:
    save_graph_nx(graph, "graph.txt")
    save_graph_nx(subgraph, "subgraph.txt")

    try:
        result = subprocess.run(
            ["../build/_deps/peregrine-src/bin/count", "graph.txt", "subgraph.txt", str(nthreads)],
            check=True,
            text=True,
            capture_output=True,
        )

        os.remove("graph.txt")
        os.remove("subgraph.txt")

        return int(result.stdout.split(":")[2].strip())
    except Exception as e:
        os.remove("graph.txt")
        os.remove("subgraph.txt")

        print(f"Error occurred while counting subgraphs {e}")
        return -1

In [ ]:
def counting_experiment(n0: int, n: int, m: int, subgraph: nx.Graph, batch_size: int, batches_number: int, nthreads: int) -> Tuple[int, int]:
    results = []
    for _ in range(batch_size * batches_number):
        graph = generate_ba_graph_nx(n0, n, m)
        if graph is None:
            print("Error generating graph")
            return -1, -1
        
        count = count_subgraphs(graph, subgraph, nthreads)
        if count == -1:
            print("Error counting subgraphs")
            return -1, -1
        
        results.append(count)

    # Central Limit Theorem usage
    means = [
        np.mean(results[i * batch_size:(i + 1) * batch_size])
        for i in range(batches_number)
    ]

    return np.mean(means), np.std(means, ddof=1) / np.sqrt(batches_number)

#### $K_3$ Counting

In [ ]:
GRAPH_N0 = 10
GRAPH_N_VALUES = list(range(50, 5051, 100))
GRAPH_M_VALUES = list(range(2, 5, 1))

K3_SUBGRAPH = nx.cycle_graph(3)

BATCH_SIZE = 100
BATCHES_NUMBER = 100
NTHREADS = 5
ALWAYS_REWRITE = True

In [ ]:
for m in GRAPH_M_VALUES:
    output_filename = f"cache/subgraphs_k3_m{m}.pkl"
    
    if os.path.exists(output_filename) and not ALWAYS_REWRITE:
        continue

    means = []
    stddevs = []
    for n in GRAPH_N_VALUES:
        print(f"Running experiment for n={n}, m={m}...")
        mean, stddev = counting_experiment(GRAPH_N0, n, m, K3_SUBGRAPH, BATCH_SIZE, BATCHES_NUMBER, NTHREADS)
        if mean == -1:
            print("Experiment failed")
            exit(1)
        
        means.append(mean)
        stddevs.append(stddev)

    with open(output_filename, "wb") as f:
        pickle.dump((GRAPH_N_VALUES, means, stddevs), f)

In [ ]:
plt.figure(figsize=(9, 5))

for m in GRAPH_M_VALUES:
    with open(f"cache/subgraphs_k3_m{m}.pkl", "rb") as f:
        ns, means, stddevs = pickle.load(f)

    plt.errorbar(ns, means, yerr=stddevs, label=f"m={m}", elinewidth=1, capsize=1, capthick=1, color=COLORS_MAPPING[m])

plt.yscale("log")
plt.xlabel("Number of nodes (n)")
plt.ylabel(r"Average number of $K_{3}$ subgraphs")
plt.title(r"Average number of $K_{3}$ subgraphs in BA graphs")
plt.grid()
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=3)
plt.tight_layout()
plt.savefig("subgraphs_k3.svg", dpi=300)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(GRAPH_M_VALUES), figsize=(18, 5))

for m, ax in zip(GRAPH_M_VALUES, [axes]):
    with open(f"cache/subgraphs_k3_m{m}.pkl", "rb") as f:
        ns, means, stddevs = pickle.load(f)

    theoretical_k5_lower_bound = [expected_number_of_cycle_subgraphs(n, m, 3, "lower_bound") for n in ns]
    theoretical_k5_upper_bound = [expected_number_of_cycle_subgraphs(n, m, 3, "upper_bound") for n in ns]

    ax.errorbar(ns, means, label="Experiment", yerr=stddevs, elinewidth=1, capsize=1, capthick=1, color=COLORS_MAPPING[m])
    ax.plot(ns, theoretical_k5_lower_bound, linestyle='--', label=f"Theoretical lower bound", color=COLORS_MAPPING[m])
    ax.plot(ns, theoretical_k5_upper_bound, linestyle='-.', label=f"Theoretical upper bound", color=COLORS_MAPPING[m])

    ax.set_yscale("log")
    ax.set_xlabel("Number of nodes (n)")
    ax.set_ylabel(r"Average number of $K_{3}$ subgraphs")
    ax.set_title(f"m={m}")
    ax.grid()
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=2)

plt.suptitle(r"Average number of $K_{3}$ subgraphs vs theory in BA graphs")
plt.tight_layout()
plt.savefig("subgraphs_k3_vs_theory.svg", dpi=300)
plt.show()

#### $K_5$ Counting

In [ ]:
GRAPH_N0 = 10
GRAPH_N_VALUES = list(range(50, 5051, 100))
GRAPH_M_VALUES = list(range(4, 7, 1))

K5_SUBGRAPH = nx.complete_graph(5)

BATCH_SIZE = 100
BATCHES_NUMBER = 100
NTHREADS = 5
ALWAYS_REWRITE = True

In [ ]:
for m in GRAPH_M_VALUES:
    output_filename = f"cache/subgraphs_k5_m{m}.pkl"
    
    if os.path.exists(output_filename) and not ALWAYS_REWRITE:
        continue

    means = []
    stddevs = []
    for n in GRAPH_N_VALUES:
        mean, stddev = counting_experiment(GRAPH_N0, n, m, K5_SUBGRAPH, BATCH_SIZE, BATCHES_NUMBER, NTHREADS)
        if mean == -1:
            print("Experiment failed")
            exit(1)
        
        means.append(mean)
        stddevs.append(stddev)

    with open(output_filename, "wb") as f:
        pickle.dump((GRAPH_N_VALUES, means, stddevs), f)

In [ ]:
plt.figure(figsize=(9, 5))

for m in GRAPH_M_VALUES:
    with open(f"cache/subgraphs_k5_m{m}.pkl", "rb") as f:
        ns, means, stddevs = pickle.load(f)

    plt.errorbar(ns, means, yerr=stddevs, label=f"m={m}", elinewidth=1, capsize=1, capthick=1, color=COLORS_MAPPING[m])

plt.yscale("log")
plt.xlabel("Number of nodes (n)")
plt.ylabel(r"Average number of $K_{5}$ subgraphs")
plt.title(r"Average number of $K_{5}$ subgraphs in BA graphs")
plt.grid()
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=3)
plt.tight_layout()
plt.savefig("subgraphs_k5.svg", dpi=300)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(GRAPH_M_VALUES), figsize=(18, 5))

for m, ax in zip(GRAPH_M_VALUES, axes):
    with open(f"cache/subgraphs_k5_m{m}.pkl", "rb") as f:
        ns, means, stddevs = pickle.load(f)

    theoretical_k5_lower_bound = [expected_number_of_clique_subgraphs(n, m, 5, "lower_bound") for n in ns]
    theoretical_k5_upper_bound = [expected_number_of_clique_subgraphs(n, m, 5, "upper_bound") for n in ns]

    ax.errorbar(ns, means, label="Experiment", yerr=stddevs, elinewidth=1, capsize=1, capthick=1, color=COLORS_MAPPING[m])
    ax.plot(ns, theoretical_k5_lower_bound, linestyle='--', label=f"Theoretical lower bound", color=COLORS_MAPPING[m])
    ax.plot(ns, theoretical_k5_upper_bound, linestyle='-.', label=f"Theoretical upper bound", color=COLORS_MAPPING[m])

    ax.set_yscale("log")
    ax.set_xlabel("Number of nodes (n)")
    ax.set_ylabel(r"Average number of $K_{5}$ subgraphs")
    ax.set_title(f"m={m}")
    ax.grid()
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=2)

plt.suptitle(r"Average number of $K_{5}$ subgraphs vs theory in BA graphs")
plt.tight_layout()
plt.savefig("subgraphs_k33_vs_theory.svg", dpi=300)
plt.show()

#### $K_{3,3}$ Counting

In [ ]:
GRAPH_N0 = 10
GRAPH_N_VALUES = list(range(50, 5051, 100))
GRAPH_M_VALUES = list(range(3, 6, 1))

K33_SUBGRAPH = nx.complete_bipartite_graph(3, 3)

BATCH_SIZE = 10
BATCHES_NUMBER = 10
NTHREADS = 5
ALWAYS_REWRITE = True

In [ ]:
for m in GRAPH_M_VALUES:
    output_filename = f"cache/subgraphs_k33_m{m}.pkl"
    if os.path.exists(output_filename) and not ALWAYS_REWRITE:
        continue

    means = []
    stddevs = []
    for n in GRAPH_N_VALUES:
        mean, stddev = counting_experiment(GRAPH_N0, n, m, K33_SUBGRAPH, BATCH_SIZE, BATCHES_NUMBER, NTHREADS)
        if mean == -1:
            print("Experiment failed")
            exit(1)
        
        means.append(mean)
        stddevs.append(stddev)

    with open(output_filename, "wb") as f:
        pickle.dump((GRAPH_N_VALUES, means, stddevs), f)

In [ ]:
plt.figure(figsize=(9, 5))

for m in GRAPH_M_VALUES:
    with open(f"cache/subgraphs_k33_m{m}.pkl", "rb") as f:
        ns, means, stddevs = pickle.load(f)

    plt.errorbar(ns, means, yerr=stddevs, label=f"m={m}", elinewidth=1, capsize=1, capthick=1, color=COLORS_MAPPING[m])

plt.yscale("log")
plt.xlabel("Number of nodes (n)")
plt.ylabel(r"Average number of $K_{3,3}$ subgraphs")
plt.title(r"Average number of $K_{3,3}$ subgraphs in BA graphs")
plt.grid()
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=3)
plt.tight_layout()
plt.savefig("subgraphs_k33.svg", dpi=300)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(GRAPH_M_VALUES), figsize=(18, 5))

for m, ax in zip(GRAPH_M_VALUES, axes):
    with open(f"cache/subgraphs_k33_m{m}.pkl", "rb") as f:
        ns, means, stddevs = pickle.load(f)

    theoretical_k33_lower_bound = [expected_number_of_bipartite_complete_subgraphs(n, m, 3, "lower_bound") for n in ns]
    theoretical_k33_upper_bound = [expected_number_of_bipartite_complete_subgraphs(n, m, 3, "upper_bound") for n in ns]

    ax.errorbar(ns, means, label="Experiment", yerr=stddevs, elinewidth=1, capsize=1, capthick=1, color=COLORS_MAPPING[m])
    ax.plot(ns, theoretical_k33_lower_bound, linestyle='--', label=f"Theoretical lower bound", color=COLORS_MAPPING[m])
    ax.plot(ns, theoretical_k33_upper_bound, linestyle='-.', label=f"Theoretical upper bound", color=COLORS_MAPPING[m])

    ax.set_yscale("log")
    ax.set_xlabel("Number of nodes (n)")
    ax.set_ylabel(r"Average number of $K_{3,3}$ subgraphs")
    ax.set_title(f"m={m}")
    ax.grid()
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=2)

plt.suptitle(r"Average number of $K_{3,3}$ subgraphs vs theory in BA graphs")
plt.tight_layout()
plt.savefig("subgraphs_k33_vs_theory.svg", dpi=300)
plt.show()